# Highlands Ranch, CO LVT Shift Notebook

This notebook follows the same overall workflow as `examples/greeley.ipynb`, but uses Douglas County parcel geometry + assessor flat files to build a Highlands Ranch-specific dataset.


In [ ]:
import io
import sys
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
import seaborn as sns
from shapely.geometry import Polygon

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "lvt_utils.py").exists():
    REPO_ROOT = REPO_ROOT.parent

if str(REPO_ROOT) not in sys.path:
    sys.path.append(str(REPO_ROOT))

from lvt_utils import calculate_category_tax_summary, model_split_rate_tax, print_category_tax_summary
from policy_analysis import analyze_land_by_improvement_share, analyze_parking_lots, analyze_vacant_land, print_parking_analysis_summary, print_vacant_land_summary
from census_utils import get_census_data_with_boundaries, match_to_census_blockgroups
from viz import create_quintile_summary

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 200)


In [ ]:
data_dir = REPO_ROOT / "examples" / "data" / "highlands_ranch"
data_dir.mkdir(parents=True, exist_ok=True)

# Douglas County ArcGIS Online org — services.arcgis.com/seTexOicoRXDvRsJ
# Discovered via putitonamap.com exploration of the county's ArcGIS org.

# PARCELSENHANCED: geometry + all key fields pre-joined, 164k county-wide rows
# Filter: CITY_NAME = 'Highlands Ranch'
PARCELS_URL = (
    "https://services.arcgis.com/seTexOicoRXDvRsJ/arcgis/rest"
    "/services/Parcels_A_view/FeatureServer/0/query"
)

# OpenData FeatureServer — non-spatial tables for valuation segments & sales
OPENDATA_BASE = (
    "https://services.arcgis.com/seTexOicoRXDvRsJ/arcgis/rest"
    "/services/OpenData/FeatureServer"
)
VALUES_URL   = OPENDATA_BASE + "/4/query"   # Property_Values_Data (301k rows)
SALES_URL    = OPENDATA_BASE + "/7/query"   # Property_Sales_Data  (676k rows)
LOCATION_URL = OPENDATA_BASE + "/5/query"   # Property_Location_Data (168k rows)

# Zoning layer — unchanged
ZONING_URL = (
    "https://services5.arcgis.com/JlofwxJO3RD8jjJH/arcgis/rest"
    "/services/Map_WFL1/FeatureServer/13/query"
)

HR_CITY   = "Highlands Ranch"
PAGE_SIZE = 2000

## Step 1: Load assessor flat files and filter to Highlands Ranch


In [ ]:
def esri_poly_to_shapely(geom):
    rings = geom.get("rings", []) if isinstance(geom, dict) else []
    if not rings:
        return None
    try:
        return Polygon(rings[0], rings[1:] if len(rings) > 1 else [])
    except Exception:
        return None


def fetch_all(url, where="1=1", out_fields="*", geometry=False,
              page_size=PAGE_SIZE, label=""):
    """Page through an ArcGIS FeatureServer layer/table and return a DataFrame.
    If geometry=True, returns a GeoDataFrame."""
    session = requests.Session()
    count_r = session.get(url, params={
        "f": "json", "where": where,
        "returnCountOnly": "true", "returnGeometry": "false",
    }, timeout=60)
    count_r.raise_for_status()
    total = count_r.json().get("count", 0)
    print(f"  {label or url.split('/')[-2]}: {total:,} rows")
    rows = []
    for offset in range(0, max(total, 1), page_size):
        r = session.get(url, params={
            "f": "json", "where": where,
            "outFields": out_fields,
            "returnGeometry": str(geometry).lower(),
            "outSR": 4326,
            "resultOffset": offset,
            "resultRecordCount": page_size,
        }, timeout=180)
        r.raise_for_status()
        feats = r.json().get("features", [])
        for feat in feats:
            a = feat.get("attributes", {})
            if geometry:
                g = esri_poly_to_shapely(feat.get("geometry", {}))
                if g:
                    a["geometry"] = g
            rows.append(a)
        if (offset // page_size) % 10 == 0 and offset > 0:
            print(f"    paged {min(offset + page_size, total):,}/{total:,}")
    if geometry:
        rows = [r for r in rows if "geometry" in r]
        return gpd.GeoDataFrame(rows, geometry="geometry", crs="EPSG:4326")
    return pd.DataFrame(rows)


# ── Step 1: Load PARCELSENHANCED (geometry + key attributes, HR only) ────
parcel_cache = data_dir / "hr_parcels_enhanced_20260401.parquet"
if parcel_cache.exists():
    print("Loading parcel cache...")
    gdf_raw = gpd.read_parquet(parcel_cache)
else:
    print("Fetching PARCELSENHANCED for Highlands Ranch...")
    gdf_raw = fetch_all(
        PARCELS_URL,
        where=f"CITY_NAME = '{HR_CITY}'",
        out_fields=(
            "STATE_PARCEL_NO,ACCOUNT_NO,CITY_NAME,ACCOUNT_TYPE_CODE,"
            "ACCOUNT_SUBTYPE_CODE,OWNER_NAME,MAILING_ADDRESS_LINE_1,"
            "MAILING_CITY_NAME,MAILING_STATE,MAILING_ZIP_CODE,"
            "ADDRESS_NUMBER,STREET_NAME,STREET_TYPE_CODE,UNIT_NO,"
            "LOCATION_ZIP_CODE,LOCATION_ADDRESS,"
            "TOTAL_ACTUAL_VALUE,TOTAL_ASSESSED_VALUE,TOTAL_NET_ACRES,"
            "SUB_FILING_RECORDING_NO,DEDICATED_SUB_FILING_NAME,"
            "PARCEL_NAME,DEEDED_AREA,CALC_AREA,BLOCK_NO,"
            "GIS_LEGAL_DESC,PARCEL_TYPE_CODE,PARCEL_TYPE,"
            "TAX_DISTRICT_NO,LAND_ECONOMIC_AREA_CODE,"
            "REDUCED_MILL_LEVY,REDUCED_TAX_RATE,"
            "LATITUDE,LONGITUDE,SHAPE__Area,SHAPE__Length"
        ),
        geometry=True,
        label="PARCELSENHANCED (HR)",
    )
    gdf_raw.to_parquet(parcel_cache, index=False)
print(f"HR parcels loaded: {len(gdf_raw):,}")

# ── Step 2: Load Property_Values_Data (segmented land/improvement) ───────
values_cache = data_dir / "property_values_data_20260401.parquet"
if values_cache.exists():
    print("Loading values cache...")
    values_df = pd.read_parquet(values_cache)
else:
    print("Fetching Property_Values_Data (full county)...")
    values_df = fetch_all(
        VALUES_URL,
        out_fields="ACCOUNT_NO,ACTUAL_VALUE,ASSESSED_VALUE,VALUATION_CLASS_CODE,VALUATION_DESCRIPTION,EXEMPT_FLAG,SUBTYPE,VALUATION_TYPE_CODE,ACCOUNT_TYPE",
        label="Property_Values_Data",
    )
    values_df.to_parquet(values_cache, index=False)

# ── Step 3: Load Property_Sales_Data ─────────────────────────────────────
sales_cache = data_dir / "property_sales_data_20260401.parquet"
if sales_cache.exists():
    print("Loading sales cache...")
    sales_df = pd.read_parquet(sales_cache)
else:
    print("Fetching Property_Sales_Data (full county)...")
    sales_df = fetch_all(
        SALES_URL,
        out_fields="ACCOUNT_NO,SALE_DATE,SALE_PRICE,DEED_TYPE,GRANTOR,GRANTEE,RECORDING_NO",
        label="Property_Sales_Data",
    )
    sales_df.to_parquet(sales_cache, index=False)

print("All sources loaded.")

In [ ]:
# ── Pivot values into land / improvement segments ─────────────────────────
hr_accts = set(gdf_raw["ACCOUNT_NO"].dropna())
values_hr = values_df[values_df["ACCOUNT_NO"].isin(hr_accts)].copy()

for col in ["ACTUAL_VALUE", "ASSESSED_VALUE"]:
    values_hr[col] = pd.to_numeric(values_hr[col], errors="coerce").fillna(0)

segs = (
    values_hr.groupby(["ACCOUNT_NO", "VALUATION_TYPE_CODE"], dropna=False)
    .agg(seg_actual=("ACTUAL_VALUE", "sum"), seg_assessed=("ASSESSED_VALUE", "sum"))
    .reset_index()
)
land_seg = (segs[segs["VALUATION_TYPE_CODE"] == "L"]
    .rename(columns={"seg_actual": "LANDACT", "seg_assessed": "LANDASD"})
    [["ACCOUNT_NO", "LANDACT", "LANDASD"]])
imp_seg  = (segs[segs["VALUATION_TYPE_CODE"] == "I"]
    .rename(columns={"seg_actual": "IMPACT", "seg_assessed": "IMPASD"})
    [["ACCOUNT_NO", "IMPACT", "IMPASD"]])

totals = values_hr.groupby("ACCOUNT_NO", dropna=False).agg(
    TOTALACT=("ACTUAL_VALUE", "sum"),
    TOTALASD=("ASSESSED_VALUE", "sum"),
    Valuation_Description=("VALUATION_DESCRIPTION", "first"),
    Valuation_Class_Code=("VALUATION_CLASS_CODE", "first"),
).reset_index()

account_values = (totals
    .merge(land_seg, on="ACCOUNT_NO", how="left")
    .merge(imp_seg,  on="ACCOUNT_NO", how="left"))
for col in ["LANDACT", "LANDASD", "IMPACT", "IMPASD"]:
    account_values[col] = pd.to_numeric(account_values[col], errors="coerce").fillna(0)

# ── Latest sale per account ───────────────────────────────────────────────
sales_df["SALE_DATE"]  = pd.to_datetime(sales_df["SALE_DATE"], unit="ms", errors="coerce")
sales_df["SALE_PRICE"] = pd.to_numeric(sales_df["SALE_PRICE"], errors="coerce")
latest_sales = (sales_df
    .sort_values(["ACCOUNT_NO", "SALE_DATE"])
    .drop_duplicates("ACCOUNT_NO", keep="last")
    [["ACCOUNT_NO", "SALE_DATE", "SALE_PRICE"]])

# ── Merge everything onto gdf_raw ────────────────────────────────────────
gdf = gdf_raw.merge(account_values, on="ACCOUNT_NO", how="left")
gdf = gdf.merge(latest_sales, on="ACCOUNT_NO", how="left")

# ── Canonical alias columns (match downstream cells) ─────────────────────
gdf["PARCEL"]    = gdf["STATE_PARCEL_NO"]
gdf["ACCOUNTNO"] = gdf["ACCOUNT_NO"]
gdf["NAME"]      = gdf["OWNER_NAME"]
gdf["LOCCITY"]   = gdf["CITY_NAME"]
gdf["SALEP"]     = gdf["SALE_PRICE"].fillna(0)
gdf["SALEDT"]    = gdf["SALE_DATE"]
gdf["ASSRCODE"]  = gdf["Valuation_Class_Code"]
gdf["ACCTTYPE"]  = gdf["ACCOUNT_TYPE_CODE"]
gdf["GIS_Acres"] = pd.to_numeric(gdf["TOTAL_NET_ACRES"], errors="coerce")
gdf["Vacant_Flag"] = gdf.get("Vacant_Flag", pd.Series("N", index=gdf.index))  # not in PARCELSENHANCED; derive below

# Vacant_Flag: derive from PARCEL_TYPE or zero improvement value
if "PARCEL_TYPE" in gdf.columns:
    gdf["Vacant_Flag"] = gdf["PARCEL_TYPE"].str.upper().str.contains("VAC", na=False).map({True:"Y", False:"N"})
else:
    gdf["Vacant_Flag"] = (gdf["IMPACT"].fillna(0) == 0).map({True:"Y", False:"N"})

for col in ["LANDACT","IMPACT","TOTALACT","LANDASD","IMPASD","TOTALASD","SALEP","GIS_Acres"]:
    gdf[col] = pd.to_numeric(gdf.get(col), errors="coerce").fillna(0)

print(f"gdf shape after attribute assembly: {gdf.shape}")
print(gdf[["PARCEL","ACCTTYPE","TOTALACT","TOTALASD","LANDACT","IMPACT"]].head(3))

In [ ]:
# Geometry is already on gdf from PARCELSENHANCED.
# Ensure CRS and add is_osm_surface_parking placeholder.
if not isinstance(gdf, gpd.GeoDataFrame) or gdf.crs is None:
    gdf = gpd.GeoDataFrame(gdf, geometry="geometry", crs="EPSG:4326")
gdf["is_osm_surface_parking"] = False
print(f"gdf shape (with geometry): {gdf.shape}")

## Step 2: Load geometry and merge


In [ ]:
# Geometry cell — PARCELSENHANCED already includes polygon geometry.
# gdf is fully assembled in the attribute cell above.
# Keeping cell as a checkpoint.
assert isinstance(gdf, gpd.GeoDataFrame), "gdf must be a GeoDataFrame"
assert "geometry" in gdf.columns, "geometry column missing"
print(f"Geometry check OK — {len(gdf):,} HR parcels with geometry")

## Step 3: Stamp zoning


In [ ]:
def load_zoning():
    cache = data_dir / "douglas_zoning_20260401.parquet"
    if cache.exists():
        return gpd.read_parquet(cache)
    session = requests.Session()
    count = session.get(zoning_query_url, params={"f":"json","where":"1=1","returnCountOnly":"true"}, timeout=60).json().get("count",0)
    feats=[]
    for off in range(0, int(count), 2000):
        r = session.get(zoning_query_url, params={
            "f":"json","where":"1=1",
            "outFields":"OBJECTID,ZONE_TYPE,FIRST_DESC",
            "returnGeometry":"true","resultOffset":off,"resultRecordCount":2000,"outSR":4326,
        }, timeout=180)
        r.raise_for_status(); feats.extend(r.json().get("features",[]))
    rows=[]
    for f in feats:
        a=f.get("attributes",{})
        g=esri_poly_to_shapely(f.get("geometry",{}))
        if g is not None:
            a["geometry"]=g; rows.append(a)
    zgdf=gpd.GeoDataFrame(rows, geometry="geometry", crs="EPSG:4326")
    zgdf.to_parquet(cache, index=False)
    return zgdf

zoning_gdf = load_zoning()
gdf_3857 = gdf.to_crs(epsg=3857)
pts = gpd.GeoDataFrame(gdf_3857.drop(columns=["geometry"]), geometry=gdf_3857.geometry.centroid, crs="EPSG:3857")
stamped = gpd.sjoin(pts, zoning_gdf.to_crs(epsg=3857)[["ZONE_TYPE","FIRST_DESC","geometry"]], how="left", predicate="within")
gdf["ZONING_CLASS"] = gdf["PARCEL"].map(stamped.groupby("PARCEL")["ZONE_TYPE"].first())
gdf["ZONING_DESC"] = gdf["PARCEL"].map(stamped.groupby("PARCEL")["FIRST_DESC"].first())


## Step 4: Categories + modeling


In [ ]:
gdf["IR"] = np.where(gdf["TOTALASD"] > 0, gdf["IMPASD"] / gdf["TOTALASD"], 0)
gdf["is_vacant"] = gdf["Vacant_Flag"].fillna("").str.upper().eq("Y")

def map_property_category(account_type, valuation_desc):
    acct = str(account_type).strip().upper()
    desc = str(valuation_desc).upper()

    if acct == "RESIDENTIAL":
        return "Residential"
    if acct == "COMMERCIAL":
        return "Commercial"
    if acct == "INDUSTRIAL":
        return "Industrial"
    if acct == "VACANT LAND" or "VAC" in desc:
        return "Vacant / Undeveloped"
    if acct in {"EXEMPT", "UTILITIES", "HOA", "POSSESSORY INT"}:
        return "Exempt / Government"
    if "AGRIC" in desc:
        return "Agricultural"
    return "Other"

gdf["PROPERTY_CATEGORY"] = gdf.apply(lambda r: map_property_category(r.get("ACCTTYPE"), r.get("Valuation_Description")), axis=1)
gdf["ANALYSIS_CATEGORY"] = np.where(gdf["ZONING_CLASS"].fillna("").eq("PD") & gdf["PROPERTY_CATEGORY"].eq("Residential"), "Residential - PD", gdf["PROPERTY_CATEGORY"])
analysis_gdf = gdf[(gdf["PROPERTY_CATEGORY"] != "Exempt / Government") & (gdf["PROPERTY_CATEGORY"] != "Agricultural")].copy()

model_df = analysis_gdf.copy()
model_df["current_tax"] = model_df["TOTALASD"] / 1000.0
current_revenue = model_df["current_tax"].sum()

scenario_outputs = {}
for ratio in [2,4,8]:
    land_mill, imp_mill, revenue, scenario_df = model_split_rate_tax(
        df=model_df.copy(),
        land_value_col="LANDASD",
        improvement_value_col="IMPASD",
        current_revenue=current_revenue,
        land_improvement_ratio=ratio,
    )
    scenario_df["tax_change"] = scenario_df["new_tax"] - scenario_df["current_tax"]
    scenario_df["tax_change_pct"] = np.where(scenario_df["current_tax"] > 0, (scenario_df["tax_change"] / scenario_df["current_tax"]) * 100, 0)
    scenario_outputs[ratio] = {"land_mill": land_mill, "imp_mill": imp_mill, "revenue": revenue, "df": scenario_df}
    print(f"{ratio}:1 split -> land mill {land_mill:.4f}, imp mill {imp_mill:.4f}, revenue {revenue:,.2f}")



## Step 5: Calculate IR, Split Tax Capacity, Run Model


In [ ]:
highlands_ranch_city = analysis_gdf.copy()
highlands_ranch_city["IR"] = np.where(highlands_ranch_city["TOTALASD"] > 0, highlands_ranch_city["IMPASD"] / highlands_ranch_city["TOTALASD"], 0)
highlands_ranch_city["TaxCapacity"] = highlands_ranch_city["TOTALASD"]
highlands_ranch_city["TaxCapacity_Improvements"] = highlands_ranch_city["IR"] * highlands_ranch_city["TaxCapacity"]
highlands_ranch_city["TaxCapacity_Land"] = (1 - highlands_ranch_city["IR"]) * highlands_ranch_city["TaxCapacity"]
highlands_ranch_city["current_tax"] = highlands_ranch_city["TOTALASD"] / 1000.0
current_revenue = highlands_ranch_city["current_tax"].sum()

print("Tax Capacity Split Summary:")
print(f"  Total Tax Capacity:          ${highlands_ranch_city['TaxCapacity'].sum():>15,.0f}")
print(f"  Tax Capacity (Improvements): ${highlands_ranch_city['TaxCapacity_Improvements'].sum():>15,.0f}")
print(f"  Tax Capacity (Land):         ${highlands_ranch_city['TaxCapacity_Land'].sum():>15,.0f}")
print(f"  Land % of Tax Capacity:      {highlands_ranch_city['TaxCapacity_Land'].sum() / highlands_ranch_city['TaxCapacity'].sum() * 100:.1f}%")

tc_land_improvement_ratio = 2
tc_land_millage, tc_imp_millage, tc_split_rate_revenue, highlands_ranch_city = model_split_rate_tax(
    df=highlands_ranch_city,
    land_value_col="TaxCapacity_Land",
    improvement_value_col="TaxCapacity_Improvements",
    current_revenue=current_revenue,
    land_improvement_ratio=tc_land_improvement_ratio,
)

highlands_ranch_city["new_tax_tc"] = highlands_ranch_city["new_tax"]
highlands_ranch_city["tax_change_tc"] = highlands_ranch_city["new_tax_tc"] - highlands_ranch_city["current_tax"]
highlands_ranch_city["tax_change_pct_tc"] = np.where(
    highlands_ranch_city["current_tax"] > 0,
    (highlands_ranch_city["tax_change_tc"] / highlands_ranch_city["current_tax"]) * 100,
    0,
)

print(f"\nFull-Bill Split-Rate Model ({tc_land_improvement_ratio}:1 ratio)")
print(f"  Land Millage:        {tc_land_millage:.6f}")
print(f"  Improvement Millage: {tc_imp_millage:.6f}")
print(f"  Current Revenue:     ${current_revenue:,.0f}")
print(f"  New Revenue:         ${highlands_ranch_city['new_tax_tc'].sum():,.0f}")
print(f"  Revenue neutral:     {abs(current_revenue - highlands_ranch_city['new_tax_tc'].sum()) < 1}")

highlands_ranch_city["LAND_DEV_CATEGORY"] = np.where(highlands_ranch_city["IR"] == 0, "Vacant Land", "Developed")
vacant_results = analyze_vacant_land(
    df=highlands_ranch_city,
    land_value_col="LANDASD",
    improvement_value_col="IMPASD",
    property_type_col="LAND_DEV_CATEGORY",
    vacant_identifier="Vacant Land",
)
underdeveloped = analyze_land_by_improvement_share(
    df=highlands_ranch_city,
    land_value_col="LANDASD",
    improvement_value_col="IMPASD",
)
total_land_value = highlands_ranch_city["LANDASD"].sum()

print("\nUndeveloped and Underdeveloped Land")
print(f"  Total non-exempt land value: ${total_land_value:,.0f}\n")
if "error" not in vacant_results:
    print("  Undeveloped (vacant, IR=0):")
    print(f"    {vacant_results['total_vacant_parcels']:,} parcels")
    print(f"    ${vacant_results['total_vacant_land_value']:,.0f} ({vacant_results['vacant_land_pct_of_total']:.1f}% of non-exempt land value)\n")

print("  Underdeveloped (by improvement share):")
for row in underdeveloped["categories"]:
    print(
        f"    {row['category']:<35} {row['parcel_count']:>7,} parcels  "
        f"${row['adjusted_land_value']:>15,.0f}  ({row['share_of_total_land_value_pct']:.1f}%)"
    )



## Step 6: Summary, diagnostics, and equity


In [ ]:
primary = scenario_outputs[4]["df"].copy()
primary["tax_change"] = primary["new_tax"] - primary["current_tax"]
primary["tax_change_pct"] = np.where(
    primary["current_tax"] > 0,
    (primary["tax_change"] / primary["current_tax"]) * 100,
    0,
)

summary = calculate_category_tax_summary(primary, "ANALYSIS_CATEGORY", "current_tax", "new_tax")
print_category_tax_summary(summary)

plot_df = (
    summary[summary["property_count"] > 50]
    .sort_values("median_tax_change_pct")
    .copy()
)

fig_height = max(5, len(plot_df) * 0.7)
plt.figure(figsize=(12, fig_height))
bar_colors = np.where(plot_df["median_tax_change_pct"] < 0, "#228B22", "#8B0000")
plt.barh(plot_df["ANALYSIS_CATEGORY"], plot_df["median_tax_change_pct"], color=bar_colors)
plt.axvline(0, color="black", linewidth=1)
plt.title("Median tax change percent by property category (4:1 split)")
plt.xlabel("Median tax change (%)")
plt.ylabel("Property category")
plt.tight_layout()
plt.show()

# Butterfly chart: percent of parcels increasing/decreasing >10%
summary_filtered = summary[summary["property_count"] > 50].copy()
summary_sorted = summary_filtered.sort_values("pct_increase_gt_threshold", ascending=True)

categories_sorted = summary_sorted["ANALYSIS_CATEGORY"].tolist()
pct_increase_sorted = summary_sorted["pct_increase_gt_threshold"].tolist()
pct_decrease_sorted = summary_sorted["pct_decrease_gt_threshold"].tolist()

pct_increase_int = [int(round(x)) for x in pct_increase_sorted]
pct_decrease_int = [int(round(x)) for x in pct_decrease_sorted]

y = np.arange(len(categories_sorted))
fig, ax = plt.subplots(figsize=(8, 6))

color_increase = "#8B0000"
color_decrease = "#228B22"

ax.barh(y, [-v for v in pct_decrease_sorted], color=color_decrease, edgecolor="none", height=0.7)
ax.barh(y, pct_increase_sorted, color=color_increase, edgecolor="none", height=0.7)

for i, (inc, dec) in enumerate(zip(pct_increase_int, pct_decrease_int)):
    if dec > 0:
        ax.text(-dec - 2, y[i], f"{dec}%", va="center", ha="right", fontsize=8, color=color_decrease)
    if inc > 0:
        ax.text(inc + 2, y[i], f"{inc}%", va="center", ha="left", fontsize=8, color=color_increase)

for i, (cat, inc) in enumerate(zip(categories_sorted, pct_increase_sorted)):
    xpos = inc + 18 if inc > 0 else 18
    ax.text(xpos, y[i], cat, va="center", ha="left", fontsize=9, fontweight="bold", color="#222")

for spine in ax.spines.values():
    spine.set_visible(False)
ax.tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)
ax.set_yticks([])
ax.set_xticks([])

max_val = max(max(pct_increase_sorted), max(pct_decrease_sorted)) if len(pct_increase_sorted) else 10
ax.set_xlim(-max_val - 20, max_val + 48)

title_y = len(categories_sorted) - 0.2
ax.text(-max_val * 0.45, title_y, "Percent of parcels
decreasing >10%", ha="center", va="bottom", fontsize=10)
ax.text(max_val * 0.45, title_y, "Percent of parcels
increasing >10%", ha="center", va="bottom", fontsize=10)

plt.tight_layout()
plt.show()

analysis_df = primary.copy()
analysis_df["LAND_USE_FOR_ANALYSIS"] = np.select([
    analysis_df["is_vacant"] == True,
    analysis_df["is_osm_surface_parking"] == True,
], ["Vacant Land", "Trans - Parking"], default="Other")

vacant_results = analyze_vacant_land(analysis_df, "LANDASD", "IMPASD", "LAND_USE_FOR_ANALYSIS", vacant_identifier="Vacant Land", owner_col="NAME")
print_vacant_land_summary(vacant_results)
parking_results = analyze_parking_lots(analysis_df, "LANDASD", "IMPASD", "LAND_USE_FOR_ANALYSIS", parking_identifier="Trans - Parking", min_land_value_threshold=50000, max_improvement_ratio=0.10)
print_parking_analysis_summary(parking_results)
display(pd.DataFrame(analyze_land_by_improvement_share(analysis_df, "LANDASD", "IMPASD")["categories"]))

try:
    _, census_boundaries = get_census_data_with_boundaries(fips_code="08035", year=2022)
    matched = match_to_census_blockgroups(primary.to_crs(epsg=4326), census_boundaries, join_type="left")
    matched = matched[matched["median_income"] > 0].copy()

    grouped = matched.groupby("std_geoid").agg(
        median_income=("median_income", "first"),
        minority_pct=("minority_pct", "first"),
        total_current_tax=("current_tax", "sum"),
        total_new_tax=("new_tax", "sum"),
        mean_tax_change=("tax_change", "mean"),
        median_tax_change=("tax_change", "median"),
        median_tax_change_pct=("tax_change_pct", "median"),
        parcel_count=("PARCEL", "count"),
        has_vacant_land=("ANALYSIS_CATEGORY", lambda x: (x == "Vacant / Undeveloped").any()),
    ).reset_index()
    grouped["mean_tax_change_pct"] = np.where(
        grouped["total_current_tax"] > 0,
        ((grouped["total_new_tax"] - grouped["total_current_tax"]) / grouped["total_current_tax"]) * 100,
        0,
    )

    non_vacant_bg = grouped[~grouped["has_vacant_land"]].copy()

    income_summary = create_quintile_summary(grouped, "median_income", "median_income", "mean_tax_change", "mean_tax_change_pct")
    minority_summary = create_quintile_summary(grouped, "minority_pct", "minority_pct", "mean_tax_change", "mean_tax_change_pct")
    non_vacant_income_summary = create_quintile_summary(non_vacant_bg, "median_income", "median_income", "mean_tax_change", "mean_tax_change_pct")
    non_vacant_minority_summary = create_quintile_summary(non_vacant_bg, "minority_pct", "minority_pct", "mean_tax_change", "mean_tax_change_pct")

    display(income_summary)
    display(non_vacant_income_summary)
    display(minority_summary)
    display(non_vacant_minority_summary)

    plt.figure(figsize=(10, 5))
    plt.plot(income_summary["median_income_quintile"], income_summary["mean_tax_change_pct"], marker="o", label="All properties")
    plt.plot(non_vacant_income_summary["median_income_quintile"], non_vacant_income_summary["mean_tax_change_pct"], marker="o", label="Excluding vacant-land neighborhoods")
    plt.axhline(0, color="black", linewidth=1, linestyle="dotted")
    plt.title("Mean tax change percent by neighborhood income quintile")
    plt.xlabel("Income quintile")
    plt.ylabel("Mean tax change (%)")
    plt.legend()
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(10, 5))
    plt.plot(minority_summary["minority_pct_quintile"], minority_summary["mean_tax_change_pct"], marker="o", label="All properties")
    plt.plot(non_vacant_minority_summary["minority_pct_quintile"], non_vacant_minority_summary["mean_tax_change_pct"], marker="o", label="Excluding vacant-land neighborhoods")
    plt.axhline(0, color="black", linewidth=1, linestyle="dotted")
    plt.title("Mean tax change percent by minority-share quintile")
    plt.xlabel("Minority-share quintile")
    plt.ylabel("Mean tax change (%)")
    plt.legend()
    plt.tight_layout()
    fig, ax = plt.subplots(figsize=(10, 6))
    vals = non_vacant_income_summary["median_tax_change_pct"]
    labels = non_vacant_income_summary["median_income_quintile"]
    colors = sns.color_palette("Greens", n_colors=len(vals))
    color_map = [colors[i] for i in np.argsort(np.argsort(-vals))]
    bars = ax.bar(labels, vals, color=color_map, edgecolor="black", width=0.7)
    ax.yaxis.set_visible(False)
    ax.set_title("Median Tax Change by Neighborhood Income (Excl. Vacant)", weight="bold", pad=30)
    sns.despine(left=True, right=True, top=True, bottom=True)
    for bar, val in zip(bars, vals):
        ax.annotate(f"{val:.1f}%", xy=(bar.get_x() + bar.get_width() / 2, val / 2), ha="center", va="center", fontsize=13, fontweight="bold")
    ax.xaxis.set_ticks_position("top")
    ax.xaxis.set_label_position("top")
    plt.xticks(fontweight="bold")
    margin = max(abs(vals.min()), abs(vals.max())) * 1.2 if len(vals) else 1
    ax.set_ylim(-margin, margin)
    ax.axhline(y=0, color="black", linewidth=0.8)
    plt.tight_layout()
    plt.show()

    fig, ax = plt.subplots(figsize=(10, 6))
    vals = non_vacant_minority_summary["median_tax_change_pct"]
    labels = non_vacant_minority_summary["minority_pct_quintile"]
    colors = sns.color_palette("Purples", n_colors=len(vals))
    color_map = [colors[i] for i in np.argsort(np.argsort(-vals))]
    bars = ax.bar(labels, vals, color=color_map, edgecolor="black", width=0.7)
    ax.yaxis.set_visible(False)
    ax.set_title("Median Tax Change by Neighborhood Minority % (Excl. Vacant)", weight="bold", pad=30)
    sns.despine(left=True, right=True, top=True, bottom=True)
    for bar, val in zip(bars, vals):
        ax.annotate(f"{val:.1f}%", xy=(bar.get_x() + bar.get_width() / 2, val / 2), ha="center", va="center", fontsize=13, fontweight="bold")
    ax.xaxis.set_ticks_position("top")
    ax.xaxis.set_label_position("top")
    plt.xticks(fontweight="bold")
    margin = max(abs(vals.min()), abs(vals.max())) * 1.2 if len(vals) else 1
    ax.set_ylim(-margin, margin)
    ax.axhline(y=0, color="black", linewidth=0.8)
    plt.tight_layout()
    plt.show()

    plt.show()

    # Residential-only census equity charts (same chart set)
    residential_matched = matched[matched["PROPERTY_CATEGORY"].eq("Residential")].copy()
    if len(residential_matched) >= 25:
        residential_grouped = residential_matched.groupby("std_geoid").agg(
            median_income=("median_income", "first"),
            minority_pct=("minority_pct", "first"),
            total_current_tax=("current_tax", "sum"),
            total_new_tax=("new_tax", "sum"),
            mean_tax_change=("tax_change", "mean"),
            median_tax_change=("tax_change", "median"),
            median_tax_change_pct=("tax_change_pct", "median"),
            parcel_count=("PARCEL", "count"),
            has_vacant_land=("ANALYSIS_CATEGORY", lambda x: (x == "Vacant / Undeveloped").any()),
        ).reset_index()
        residential_grouped["mean_tax_change_pct"] = np.where(
            residential_grouped["total_current_tax"] > 0,
            ((residential_grouped["total_new_tax"] - residential_grouped["total_current_tax"]) / residential_grouped["total_current_tax"]) * 100,
            0,
        )

        residential_non_vacant = residential_grouped[~residential_grouped["has_vacant_land"]].copy()

        res_income_summary = create_quintile_summary(residential_grouped, "median_income", "median_income", "mean_tax_change", "mean_tax_change_pct")
        res_minority_summary = create_quintile_summary(residential_grouped, "minority_pct", "minority_pct", "mean_tax_change", "mean_tax_change_pct")
        res_non_vacant_income_summary = create_quintile_summary(residential_non_vacant, "median_income", "median_income", "mean_tax_change", "mean_tax_change_pct")
        res_non_vacant_minority_summary = create_quintile_summary(residential_non_vacant, "minority_pct", "minority_pct", "mean_tax_change", "mean_tax_change_pct")

        display(res_income_summary)
        display(res_non_vacant_income_summary)
        display(res_minority_summary)
        display(res_non_vacant_minority_summary)

        plt.figure(figsize=(10, 5))
        plt.plot(res_income_summary["median_income_quintile"], res_income_summary["mean_tax_change_pct"], marker="o", label="Residential only")
        plt.plot(res_non_vacant_income_summary["median_income_quintile"], res_non_vacant_income_summary["mean_tax_change_pct"], marker="o", label="Residential (excl. vacant-land neighborhoods)")
        plt.axhline(0, color="black", linewidth=1, linestyle="dotted")
        plt.title("Residential-only mean tax change percent by neighborhood income quintile")
        plt.xlabel("Income quintile")
        plt.ylabel("Mean tax change (%)")
        plt.legend()
        plt.tight_layout()
        plt.show()

        plt.figure(figsize=(10, 5))
        plt.plot(res_minority_summary["minority_pct_quintile"], res_minority_summary["mean_tax_change_pct"], marker="o", label="Residential only")
        plt.plot(res_non_vacant_minority_summary["minority_pct_quintile"], res_non_vacant_minority_summary["mean_tax_change_pct"], marker="o", label="Residential (excl. vacant-land neighborhoods)")
        plt.axhline(0, color="black", linewidth=1, linestyle="dotted")
        plt.title("Residential-only mean tax change percent by minority-share quintile")
        plt.xlabel("Minority-share quintile")
        plt.ylabel("Mean tax change (%)")
        plt.legend()
        plt.tight_layout()
        plt.show()

        fig, ax = plt.subplots(figsize=(10, 6))
        vals = res_non_vacant_income_summary["median_tax_change_pct"]
        labels = res_non_vacant_income_summary["median_income_quintile"]
        colors = sns.color_palette("Greens", n_colors=len(vals))
        color_map = [colors[i] for i in np.argsort(np.argsort(-vals))]
        bars = ax.bar(labels, vals, color=color_map, edgecolor="black", width=0.7)
        ax.yaxis.set_visible(False)
        ax.set_title("Residential Median Tax Change by Neighborhood Income (Excl. Vacant)", weight="bold", pad=30)
        sns.despine(left=True, right=True, top=True, bottom=True)
        for bar, val in zip(bars, vals):
            ax.annotate(f"{val:.1f}%", xy=(bar.get_x() + bar.get_width() / 2, val / 2), ha="center", va="center", fontsize=13, fontweight="bold")
        ax.xaxis.set_ticks_position("top")
        ax.xaxis.set_label_position("top")
        plt.xticks(fontweight="bold")
        margin = max(abs(vals.min()), abs(vals.max())) * 1.2 if len(vals) else 1
        ax.set_ylim(-margin, margin)
        ax.axhline(y=0, color="black", linewidth=0.8)
        plt.tight_layout()
        plt.show()

        fig, ax = plt.subplots(figsize=(10, 6))
        vals = res_non_vacant_minority_summary["median_tax_change_pct"]
        labels = res_non_vacant_minority_summary["minority_pct_quintile"]
        colors = sns.color_palette("Purples", n_colors=len(vals))
        color_map = [colors[i] for i in np.argsort(np.argsort(-vals))]
        bars = ax.bar(labels, vals, color=color_map, edgecolor="black", width=0.7)
        ax.yaxis.set_visible(False)
        ax.set_title("Residential Median Tax Change by Neighborhood Minority % (Excl. Vacant)", weight="bold", pad=30)
        sns.despine(left=True, right=True, top=True, bottom=True)
        for bar, val in zip(bars, vals):
            ax.annotate(f"{val:.1f}%", xy=(bar.get_x() + bar.get_width() / 2, val / 2), ha="center", va="center", fontsize=13, fontweight="bold")
        ax.xaxis.set_ticks_position("top")
        ax.xaxis.set_label_position("top")
        plt.xticks(fontweight="bold")
        margin = max(abs(vals.min()), abs(vals.max())) * 1.2 if len(vals) else 1
        ax.set_ylim(-margin, margin)
        ax.axhline(y=0, color="black", linewidth=0.8)
        plt.tight_layout()
        plt.show()
    else:
        print("Residential-only census charts skipped: not enough matched residential parcels.")

except Exception as exc:
    print(f"Census section skipped: {exc}")



